# 17 — Mitigation de la concentration exponentielle : kernels locaux et multi-echelles

Implementation et reproduction des resultats de :
> **Local and Multi-Scale Strategies to Mitigate Exponential Concentration in Quantum Kernels**  
> Zendejas-Morales, Saikia, Singh — arXiv 2602.16097 (2026)

| Analyse | Description |
|---------|-------------|
| **1** | Concentration du kernel global (baseline) en fonction de Q |
| **2** | Kernels locaux par patches — mitigation de la concentration |
| **3** | Kernels multi-echelles — combinaison de granularites |
| **4** | Comparaison complete : concentration, richesse spectrale, CKA, AUC |

**Hypothese principale du papier** : la concentration exponentielle n'implique pas de mauvaise performance, mais reduit la richesse informationnelle du kernel. La mitigation geometrique ne garantit pas un gain en AUC sans alignement avec les labels.

**Configuration** : N=120, Q=4..12, datasets : Breast Cancer + German Credit + Bank Marketing

In [ ]:
import sys, warnings, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path('.').resolve().parent
sys.path.insert(0, str(ROOT))

from data.loaders import load_dataset
from src.preprocessing import QuantumScaler, FeatureReducer

OUT = ROOT / 'results' / '17'
OUT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'savefig.bbox': 'tight',
})

N_SAMPLES = 120
SEED      = 42
D_VALUES  = [4, 6, 8, 10, 12]   # nombre de features / qubits

print(f'Configuration : N={N_SAMPLES}, SEED={SEED}')
print(f'Dimensions    : {D_VALUES}')
print(f'Resultats -> {OUT}')

---
## Fonctions utilitaires

Trois types de kernels + quatre metriques de diagnostic (cf. papier section 3).

In [ ]:
from qiskit.circuit.library import PauliFeatureMap
from qiskit.quantum_info import Statevector
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# ------------------------------------------------------------------ #
#  Kernel fidelite global (baseline du papier)                        #
# ------------------------------------------------------------------ #
def statevectors(fm, X):
    """Calcule les statevectors pour chaque point de X."""
    params = list(fm.parameters)
    dim    = 2 ** fm.num_qubits
    svs    = np.zeros((len(X), dim), dtype=complex)
    for i, x in enumerate(X):
        svs[i] = Statevector.from_instruction(
            fm.assign_parameters({p: float(v) for p, v in zip(params, x)})
        ).data
    return svs

def fidelity_kernel_global(fm, X):
    """K^fid_ij = |<psi(xi)|psi(xj)>|^2 (eq. 1 du papier)."""
    svs = statevectors(fm, X)
    return np.abs(svs @ svs.conj().T) ** 2

# ------------------------------------------------------------------ #
#  Kernel local par patches (section 4.1)                             #
# ------------------------------------------------------------------ #
def fidelity_kernel_local(X, Q, patch_size, alpha=1.0):
    """
    Decoupe Q qubits en patches de taille patch_size.
    Chaque patch utilise les features correspondantes.
    K_local = moyenne des K_patch (poids uniformes).
    """
    n_patches = Q // patch_size
    K_acc = np.zeros((len(X), len(X)))
    for m in range(n_patches):
        feat_idx = slice(m * patch_size, (m + 1) * patch_size)
        X_patch  = X[:, feat_idx]
        fm_patch = PauliFeatureMap(patch_size, reps=1,
                                   paulis=['Z', 'ZZ'] if patch_size >= 2 else ['Z'],
                                   alpha=alpha, entanglement='linear')
        K_acc += fidelity_kernel_global(fm_patch, X_patch)
    return K_acc / n_patches

# ------------------------------------------------------------------ #
#  Kernel multi-echelles (section 4.2)                                #
# ------------------------------------------------------------------ #
def fidelity_kernel_multiscale(X, Q, patch_sizes, alpha=1.0):
    """
    Combine des kernels locaux a plusieurs granularites.
    K_ms = moyenne sur les echelles (poids uniformes).
    patch_sizes : liste de tailles de patch valides pour Q.
    """
    valid_sizes = [p for p in patch_sizes if Q % p == 0 and p <= Q]
    if not valid_sizes:
        return fidelity_kernel_global(
            PauliFeatureMap(Q, reps=1, paulis=['Z','ZZ'], alpha=alpha, entanglement='linear'), X)
    K_acc = np.zeros((len(X), len(X)))
    for ps in valid_sizes:
        K_acc += fidelity_kernel_local(X, Q, ps, alpha=alpha)
    return K_acc / len(valid_sizes)

# ------------------------------------------------------------------ #
#  Metriques de diagnostic (section 3 du papier)                      #
# ------------------------------------------------------------------ #
def concentration_stats(K):
    """p50 et p95 des elements hors-diagonale (papier eq. 3-4)."""
    n    = K.shape[0]
    mask = ~np.eye(n, dtype=bool)
    off  = K[mask]
    return float(np.percentile(off, 50)), float(np.percentile(off, 95))

def spectral_entropy(K):
    """Entropie normalisee du spectre : H/H_max (richesse spectrale)."""
    eigvals = np.maximum(np.linalg.eigvalsh(K), 1e-12)
    p       = eigvals / eigvals.sum()
    H       = float(-np.sum(p * np.log(p)))
    H_max   = np.log(len(eigvals))
    return H / H_max if H_max > 0 else 0.0

def centered_kernel_alignment(K, y):
    """CKA entre K et le kernel cible yy^T (alignement avec les labels)."""
    Kt  = (y[:, None] == y[None, :]).astype(float)
    n   = len(y)
    H   = np.eye(n) - np.ones((n, n)) / n
    Kc  = H @ K  @ H
    Ktc = H @ Kt @ H
    num = np.trace(Kc @ Ktc)
    den = np.linalg.norm(Kc, 'fro') * np.linalg.norm(Ktc, 'fro') + 1e-24
    return float(num / den)

def make_psd(K, eps=1e-8):
    lam = np.linalg.eigvalsh(K).min()
    if lam < 0:
        K = K + (abs(lam) + eps) * np.eye(K.shape[0])
    return K

def eval_svm_auc(K, y, n_splits=15, seed=42):
    """AUC moyenne sur n_splits avec SVM precompute."""
    K = make_psd(K)
    aucs = []
    for r in range(n_splits):
        tr, te = train_test_split(np.arange(len(y)), test_size=0.30,
                                  random_state=seed + r, stratify=y)
        svm = SVC(kernel='precomputed', C=1.0, probability=True)
        svm.fit(K[np.ix_(tr, tr)], y[tr])
        try:
            aucs.append(roc_auc_score(y[te],
                svm.predict_proba(K[np.ix_(te, tr)])[:, 1]))
        except Exception:
            aucs.append(0.5)
    return float(np.mean(aucs)), float(np.std(aucs, ddof=1))

print('Fonctions chargees : fidelity_kernel_global, _local, _multiscale')
print('Metriques : concentration_stats, spectral_entropy, centered_kernel_alignment, eval_svm_auc')

---
## Analyse 1 — Concentration du kernel global en fonction de Q

Reproduction de la figure 1 du papier : le kernel de fidelite global montre une concentration
exponentielle — les valeurs hors-diagonale de la matrice de Gram s'effondrent vers 0 quand Q augmente.

On mesure p50 et p95 des elements hors-diagonale sur les 3 datasets du projet.

In [ ]:
DATASETS = [
    ('breast_cancer',  'Breast Cancer'),
    ('german_credit',  'German Credit'),
    ('bank_marketing', 'Bank Marketing'),
]
ALPHA = 1.0

# Resultats : pour chaque dataset et chaque Q, stocker p50/p95/H
results_global = {ds: {'p50': [], 'p95': [], 'H': [], 'cka': []} for ds, _ in DATASETS}

print(f'Sweep concentration globale : D_VALUES={D_VALUES}, alpha={ALPHA}')
t0 = time.time()

for ds_name, ds_label in DATASETS:
    print(f'\n  {ds_label}')
    X_raw, y = load_dataset(ds_name, n_samples=N_SAMPLES, random_state=SEED)

    for Q in D_VALUES:
        reducer = FeatureReducer(n_components=Q)
        scaler  = QuantumScaler(feature_range=(0, 2))
        X_proc  = scaler.fit_transform(reducer.fit_transform(X_raw))

        fm  = PauliFeatureMap(Q, reps=1, paulis=['Z', 'ZZ'], alpha=ALPHA, entanglement='linear')
        K   = fidelity_kernel_global(fm, X_proc)

        p50, p95 = concentration_stats(K)
        H        = spectral_entropy(K)
        cka      = centered_kernel_alignment(K, y)

        results_global[ds_name]['p50'].append(p50)
        results_global[ds_name]['p95'].append(p95)
        results_global[ds_name]['H'].append(H)
        results_global[ds_name]['cka'].append(cka)

        print(f'    Q={Q:2d}  p50={p50:.4f}  p95={p95:.4f}  H/H_max={H:.3f}  CKA={cka:.4f}')

print(f'\nTotal : {time.time()-t0:.1f}s')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors_ds = ['#e74c3c', '#2980b9', '#27ae60']
markers   = ['o', 's', '^']

metrics = [('p50', 'Mediane off-diag (p50)', 'Concentration croissante -> 0'),
           ('p95', 'Percentile 95 off-diag (p95)', 'Concentration croissante -> 0'),
           ('H',   'Entropie spectrale H/H_max', 'Richesse spectrale (1=expressif)')]

for ax, (metric, ylabel, subtitle) in zip(axes, metrics):
    for (ds_name, ds_label), c, mk in zip(DATASETS, colors_ds, markers):
        vals = results_global[ds_name][metric]
        ax.plot(D_VALUES, vals, marker=mk, color=c, lw=2, ms=7, label=ds_label)
    ax.set_xlabel('Nombre de qubits / features Q')
    ax.set_ylabel(ylabel)
    ax.set_title(f'Kernel global\n{subtitle}', fontweight='bold')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=9)
    ax.set_xticks(D_VALUES)

plt.suptitle('A1 — Concentration exponentielle du kernel global (baseline)\n'
             'Reproduction figure 1, Zendejas-Morales et al. 2602.16097',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(OUT / '17_A1_global_concentration.png', dpi=150)
plt.show()
print('A1 sauvegarde')

---
## Analyse 2 — Kernels locaux par patches

Implementation de l'**algorithme 1** du papier : les Q qubits sont partitionnes en patches
de taille `patch_size`. Le kernel local est la moyenne des kernels de fidelite sur chaque patch.

On compare trois patch sizes (1, 2, 4) contre le kernel global pour Q=8 et Q=12.

In [ ]:
# Sweep patch_size x Q pour Breast Cancer (dataset du papier)
PATCH_SIZES = [1, 2, 4]
Q_TEST      = [4, 6, 8, 10, 12]

# Pour chaque (Q, patch_size), calcul des metriques
res_local = {ps: {'p50': [], 'p95': [], 'H': [], 'cka': []} for ps in PATCH_SIZES}
res_local['global'] = {'p50': [], 'p95': [], 'H': [], 'cka': []}

X_raw_bc, y_bc = load_dataset('breast_cancer', n_samples=N_SAMPLES, random_state=SEED)

print('Sweep kernels locaux (Breast Cancer, patch_sizes={}, Q={})'.format(PATCH_SIZES, Q_TEST))
t0 = time.time()

for Q in Q_TEST:
    reducer = FeatureReducer(n_components=Q)
    scaler  = QuantumScaler(feature_range=(0, 2))
    X_proc  = scaler.fit_transform(reducer.fit_transform(X_raw_bc))

    # Global baseline
    fm_g = PauliFeatureMap(Q, reps=1, paulis=['Z', 'ZZ'], alpha=ALPHA, entanglement='linear')
    K_g  = fidelity_kernel_global(fm_g, X_proc)
    p50g, p95g = concentration_stats(K_g)
    res_local['global']['p50'].append(p50g)
    res_local['global']['p95'].append(p95g)
    res_local['global']['H'].append(spectral_entropy(K_g))
    res_local['global']['cka'].append(centered_kernel_alignment(K_g, y_bc))

    # Local kernels pour chaque patch size valide
    for ps in PATCH_SIZES:
        if Q % ps == 0:
            K_l = fidelity_kernel_local(X_proc, Q, ps, alpha=ALPHA)
            p50, p95 = concentration_stats(K_l)
            res_local[ps]['p50'].append(p50)
            res_local[ps]['p95'].append(p95)
            res_local[ps]['H'].append(spectral_entropy(K_l))
            res_local[ps]['cka'].append(centered_kernel_alignment(K_l, y_bc))
        else:
            # Q non divisible par ps : NaN
            res_local[ps]['p50'].append(np.nan)
            res_local[ps]['p95'].append(np.nan)
            res_local[ps]['H'].append(np.nan)
            res_local[ps]['cka'].append(np.nan)

    # Log
    valid = [ps for ps in PATCH_SIZES if Q % ps == 0]
    info  = '  '.join([f'p{ps}:p50={res_local[ps]["p50"][-1]:.4f}' for ps in valid])
    print(f'  Q={Q:2d}  global:p50={p50g:.4f}  {info}')

print(f'\nTotal : {time.time()-t0:.1f}s')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_patch = {'global': '#e74c3c', 1: '#8e44ad', 2: '#2980b9', 4: '#27ae60'}
labels_patch = {'global': 'Global (baseline)', 1: 'Local p=1', 2: 'Local p=2', 4: 'Local p=4'}
markers_p    = {'global': 'o', 1: 's', 2: '^', 4: 'D'}

for ax, (metric, title) in zip(axes, [('p50', 'Mediane off-diag (p50)'), ('H', 'Entropie spectrale H/H_max')]):
    for key in ['global'] + PATCH_SIZES:
        vals = res_local[key][metric]
        # Filtrer les NaN pour le trace
        q_valid = [Q_TEST[i] for i, v in enumerate(vals) if not np.isnan(v)]
        v_valid = [v          for v   in vals           if not np.isnan(v)]
        ax.plot(q_valid, v_valid, marker=markers_p[key], color=colors_patch[key],
                lw=2, ms=8, label=labels_patch[key])
    ax.set_xlabel('Nombre de qubits Q')
    ax.set_ylabel(title)
    ax.set_title(f'Kernels locaux vs global\n{title}', fontweight='bold')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=9)
    ax.set_xticks(Q_TEST)

plt.suptitle('A2 — Mitigation par kernels locaux (Breast Cancer, N=120)\n'
             'Algorithme 1 — Zendejas-Morales et al. 2602.16097',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(OUT / '17_A2_local_kernels.png', dpi=150)
plt.show()
print('A2 sauvegarde')

---
## Analyse 3 — Kernels multi-echelles

Implementation de l'**algorithme 2** du papier : on combine des kernels locaux de plusieurs
granularites (patch sizes) avec des poids uniformes.

```
K_ms = (1/S) * sum_s K_local^(s)   s in {1, 2, 4}
```

Comparaison directe global vs local (meilleur) vs multi-echelle sur les 3 datasets.

In [ ]:
MS_PATCH_SIZES = [1, 2, 4]   # echelles combinees

# Structure : dataset x methode x Q -> metriques
methods_ms = ['global', 'local_p2', 'multiscale']
res_ms = {
    ds_name: {m: {'p50': [], 'p95': [], 'H': [], 'cka': [], 'auc_m': [], 'auc_s': []}
              for m in methods_ms}
    for ds_name, _ in DATASETS
}

print('Sweep global vs local vs multi-echelles (3 datasets x {} Q)'.format(len(D_VALUES)))
t0 = time.time()

for ds_name, ds_label in DATASETS:
    print(f'\n  {ds_label}')
    X_raw, y = load_dataset(ds_name, n_samples=N_SAMPLES, random_state=SEED)

    for Q in D_VALUES:
        reducer = FeatureReducer(n_components=Q)
        scaler  = QuantumScaler(feature_range=(0, 2))
        X_proc  = scaler.fit_transform(reducer.fit_transform(X_raw))

        # --- Global ---
        fm_g = PauliFeatureMap(Q, reps=1, paulis=['Z', 'ZZ'], alpha=ALPHA, entanglement='linear')
        K_g  = fidelity_kernel_global(fm_g, X_proc)

        # --- Local p=2 (meilleure mitigation pour Q pair) ---
        ps_best = 2 if Q % 2 == 0 else 1
        K_l = fidelity_kernel_local(X_proc, Q, ps_best, alpha=ALPHA)

        # --- Multi-echelles ---
        K_ms = fidelity_kernel_multiscale(X_proc, Q, MS_PATCH_SIZES, alpha=ALPHA)

        for meth, K in zip(methods_ms, [K_g, K_l, K_ms]):
            p50, p95 = concentration_stats(K)
            H        = spectral_entropy(K)
            cka      = centered_kernel_alignment(K, y)
            auc_m, auc_s = eval_svm_auc(K, y, n_splits=10, seed=SEED)
            res_ms[ds_name][meth]['p50'].append(p50)
            res_ms[ds_name][meth]['p95'].append(p95)
            res_ms[ds_name][meth]['H'].append(H)
            res_ms[ds_name][meth]['cka'].append(cka)
            res_ms[ds_name][meth]['auc_m'].append(auc_m)
            res_ms[ds_name][meth]['auc_s'].append(auc_s)

        print(f'    Q={Q:2d}  '
              f'glob:p50={res_ms[ds_name]["global"]["p50"][-1]:.4f} '
              f'loc:p50={res_ms[ds_name]["local_p2"]["p50"][-1]:.4f} '
              f'ms:p50={res_ms[ds_name]["multiscale"]["p50"][-1]:.4f}  '
              f'AUC glob={res_ms[ds_name]["global"]["auc_m"][-1]:.3f} '
              f'ms={res_ms[ds_name]["multiscale"]["auc_m"][-1]:.3f}')

print(f'\nTotal : {time.time()-t0:.1f}s')

In [ ]:
colors_m  = {'global': '#e74c3c', 'local_p2': '#2980b9', 'multiscale': '#27ae60'}
labels_m  = {'global': 'Global (baseline)', 'local_p2': 'Local p=2', 'multiscale': 'Multi-echelles {1,2,4}'}
markers_m = {'global': 'o', 'local_p2': 's', 'multiscale': 'D'}
lss       = {'global': '-', 'local_p2': '--', 'multiscale': '-.'}

fig, axes = plt.subplots(3, 4, figsize=(22, 14))
metric_defs = [
    ('p50', 'p50 off-diag\n(concentration, -> 0 = mauvais)'),
    ('H',   'H/H_max\n(richesse spectrale, 1=max)'),
    ('cka', 'CKA avec labels\n(alignement, plus = mieux)'),
    ('auc_m', 'AUC SVM (10 runs)\n(performance aval)'),
]

for row_idx, (ds_name, ds_label) in enumerate(DATASETS):
    for col_idx, (metric, ylabel) in enumerate(metric_defs):
        ax = axes[row_idx, col_idx]
        for meth in methods_ms:
            vals = res_ms[ds_name][meth][metric]
            ax.plot(D_VALUES, vals, marker=markers_m[meth], color=colors_m[meth],
                    lw=2, ms=7, ls=lss[meth], label=labels_m[meth])
            if metric == 'auc_m':
                stds = res_ms[ds_name][meth]['auc_s']
                ax.fill_between(D_VALUES,
                                [m - s for m, s in zip(vals, stds)],
                                [m + s for m, s in zip(vals, stds)],
                                alpha=0.12, color=colors_m[meth])
        ax.set_xlabel('Q' if row_idx == 2 else '')
        ax.set_ylabel(ylabel if col_idx == 0 else '')
        if row_idx == 0:
            ax.set_title(ylabel.split('\n')[0], fontweight='bold')
        if col_idx == 0:
            ax.set_ylabel(f'{ds_label}\n{ylabel}', fontsize=9)
        ax.grid(alpha=0.3)
        ax.set_xticks(D_VALUES)
        if row_idx == 0 and col_idx == 0:
            ax.legend(fontsize=8)

plt.suptitle('A3 — Global vs Local vs Multi-echelles : 4 metriques x 3 datasets\n'
             'Zendejas-Morales et al. 2602.16097 — datasets financiers',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUT / '17_A3_multiscale_comparison.png', dpi=150)
plt.show()
print('A3 sauvegarde')

---
## Analyse 4 — Test de la these centrale du papier

> *"Reduced concentration does not necessarily imply higher accuracy in a dataset-independent way.  
>  A kernel that is less concentrated can still fail to improve test accuracy if the preserved  
>  variation is not aligned with label structure."*

On quantifie directement la relation entre concentration mitiguee et gain en AUC via :

1. **Scatter** : delta_concentration vs delta_AUC (multi-echelles - global)
2. **Heatmap** : p50 vs CKA (geometrique vs alignement)
3. **Corr. de Pearson** entre p50, H, CKA et AUC sur tous les points (3 datasets x 5 Q)

In [ ]:
# Construire un tableau plat de toutes les observations
records = []
for ds_name, ds_label in DATASETS:
    for qi, Q in enumerate(D_VALUES):
        for meth in methods_ms:
            records.append({
                'dataset':  ds_label,
                'Q':        Q,
                'method':   meth,
                'p50':      res_ms[ds_name][meth]['p50'][qi],
                'p95':      res_ms[ds_name][meth]['p95'][qi],
                'H':        res_ms[ds_name][meth]['H'][qi],
                'cka':      res_ms[ds_name][meth]['cka'][qi],
                'auc':      res_ms[ds_name][meth]['auc_m'][qi],
            })

# Deltas multi-echelles - global pour chaque (dataset, Q)
delta_records = []
for ds_name, ds_label in DATASETS:
    for qi, Q in enumerate(D_VALUES):
        d_p50 = res_ms[ds_name]['multiscale']['p50'][qi] - res_ms[ds_name]['global']['p50'][qi]
        d_H   = res_ms[ds_name]['multiscale']['H'][qi]   - res_ms[ds_name]['global']['H'][qi]
        d_cka = res_ms[ds_name]['multiscale']['cka'][qi] - res_ms[ds_name]['global']['cka'][qi]
        d_auc = res_ms[ds_name]['multiscale']['auc_m'][qi] - res_ms[ds_name]['global']['auc_m'][qi]
        delta_records.append({'dataset': ds_label, 'Q': Q,
                               'd_p50': d_p50, 'd_H': d_H, 'd_cka': d_cka, 'd_auc': d_auc})

d_p50_all = np.array([r['d_p50'] for r in delta_records])
d_H_all   = np.array([r['d_H']   for r in delta_records])
d_cka_all = np.array([r['d_cka'] for r in delta_records])
d_auc_all = np.array([r['d_auc'] for r in delta_records])
ds_colors  = [colors_ds[i] for r in delta_records
               for i, (dn, _) in enumerate(DATASETS) if r['dataset'] == _]

# Correlations sur tous les points (methodes poolees)
p50_all  = np.array([r['p50']  for r in records])
H_all    = np.array([r['H']    for r in records])
cka_all  = np.array([r['cka']  for r in records])
auc_all  = np.array([r['auc']  for r in records])

print('Correlations (toutes methodes, tous datasets, tous Q):')
for name, arr in [('p50', p50_all), ('H/H_max', H_all), ('CKA', cka_all)]:
    r = np.corrcoef(arr, auc_all)[0, 1]
    print(f'  corr({name}, AUC) = {r:+.4f}')

print('\nCorrelations des deltas (multiscale - global):')
for name, arr in [('delta_p50', d_p50_all), ('delta_H', d_H_all), ('delta_CKA', d_cka_all)]:
    r = np.corrcoef(arr, d_auc_all)[0, 1]
    print(f'  corr({name}, delta_AUC) = {r:+.4f}')

In [ ]:
from scipy.stats import pearsonr

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
ds_color_map = {dl: c for (_, dl), c in zip(DATASETS, colors_ds)}

# --- Scatter 1 : delta_p50 vs delta_AUC ---
ax = axes[0]
for r in delta_records:
    ax.scatter(r['d_p50'], r['d_auc'],
               color=ds_color_map[r['dataset']], s=70, zorder=3,
               label=r['dataset'] if r['Q'] == D_VALUES[0] else '')
    ax.annotate(f"Q={r['Q']}", (r['d_p50'], r['d_auc']),
                textcoords='offset points', xytext=(4, 3), fontsize=7, alpha=0.7)
ax.axhline(0, color='grey', ls=':', lw=1)
ax.axvline(0, color='grey', ls=':', lw=1)
r_val, p_val = pearsonr(d_p50_all, d_auc_all)
ax.set_xlabel('delta p50 (multi-echelles - global)')
ax.set_ylabel('delta AUC (multi-echelles - global)')
ax.set_title(f'Mitigation geometrique vs gain AUC\nr={r_val:+.3f}  p={p_val:.3f}',
             fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# --- Scatter 2 : CKA vs AUC (tous points) ---
ax = axes[1]
method_mks = {'global': 'o', 'local_p2': 's', 'multiscale': 'D'}
for r in records:
    ax.scatter(r['cka'], r['auc'],
               color=ds_color_map[r['dataset']],
               marker=method_mks[r['method']], s=55, zorder=3, alpha=0.8)
r_cka, p_cka = pearsonr(cka_all, auc_all)
z = np.polyfit(cka_all, auc_all, 1)
x_line = np.linspace(cka_all.min(), cka_all.max(), 50)
ax.plot(x_line, np.polyval(z, x_line), 'k--', lw=1.5, alpha=0.6)
ax.set_xlabel('CKA avec labels')
ax.set_ylabel('AUC (SVM, 10 runs)')
ax.set_title(f'Alignement label (CKA) vs AUC\nr={r_cka:+.3f}  p={p_cka:.3f}',
             fontweight='bold')
legend_elements = [
    mpatches.Patch(color=c, label=dl) for (_, dl), c in zip(DATASETS, colors_ds)
]
ax.legend(handles=legend_elements, fontsize=9)
ax.grid(alpha=0.3)

# --- Bar : correlation de chaque metrique avec AUC ---
ax = axes[2]
metric_names = ['p50', 'H/H_max', 'CKA']
corrs = [np.corrcoef(p50_all, auc_all)[0, 1],
         np.corrcoef(H_all,   auc_all)[0, 1],
         np.corrcoef(cka_all, auc_all)[0, 1]]
bar_colors = ['#e74c3c', '#f39c12', '#27ae60']
bars = ax.barh(metric_names, corrs, color=bar_colors, alpha=0.8)
ax.axvline(0, color='black', lw=1)
for bar, val in zip(bars, corrs):
    ax.text(val + (0.01 if val >= 0 else -0.01), bar.get_y() + bar.get_height()/2,
            f'{val:+.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=10)
ax.set_xlabel('Correlation de Pearson avec AUC')
ax.set_title('Predictivite des metriques\npour la performance aval', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
ax.set_xlim(-0.8, 0.8)

plt.suptitle('A4 — Test de la these du papier :\n'
             'mitigation geometrique vs gain AUC (non garanti sans alignement label)',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(OUT / '17_A4_thesis_test.png', dpi=150)
plt.show()
print('A4 sauvegarde')

---
## Analyse 5 — Spectres des valeurs propres

Visualisation directe des spectres de Gram pour Q=8 (Breast Cancer) :
le kernel global a un spectre fortement decroissant (rankeffectif bas),
les kernels locaux et multi-echelles ont des spectres plus plats.

In [ ]:
Q_SPEC = 8   # Q fixe pour visualisation spectrale

X_raw_bc2, y_bc2 = load_dataset('breast_cancer', n_samples=N_SAMPLES, random_state=SEED)
reducer_sp = FeatureReducer(n_components=Q_SPEC)
scaler_sp  = QuantumScaler(feature_range=(0, 2))
X_sp       = scaler_sp.fit_transform(reducer_sp.fit_transform(X_raw_bc2))

print(f'Calcul des spectres pour Q={Q_SPEC}, Breast Cancer...')
t0 = time.time()

# Kernels
fm_sp  = PauliFeatureMap(Q_SPEC, reps=1, paulis=['Z','ZZ'], alpha=ALPHA, entanglement='linear')
K_sp_g = fidelity_kernel_global(fm_sp, X_sp)
K_sp_l = fidelity_kernel_local(X_sp,  Q_SPEC, patch_size=2, alpha=ALPHA)
K_sp_m = fidelity_kernel_multiscale(X_sp, Q_SPEC, [1, 2, 4], alpha=ALPHA)

# Spectres
def get_spectrum(K):
    eigs = np.sort(np.linalg.eigvalsh(K))[::-1]
    return eigs / eigs.sum()

spec_g = get_spectrum(K_sp_g)
spec_l = get_spectrum(K_sp_l)
spec_m = get_spectrum(K_sp_m)

# Rang effectif (Johnson 2019 : sum / max)
def eff_rank(K):
    eigs = np.maximum(np.linalg.eigvalsh(K), 0)
    p    = eigs / (eigs.sum() + 1e-24)
    return float(np.exp(-np.sum(p[p > 0] * np.log(p[p > 0]))))

er_g = eff_rank(K_sp_g)
er_l = eff_rank(K_sp_l)
er_m = eff_rank(K_sp_m)

print(f'  Rang effectif global      : {er_g:.2f}')
print(f'  Rang effectif local p=2   : {er_l:.2f}')
print(f'  Rang effectif multiscale  : {er_m:.2f}')
print(f'  Total : {time.time()-t0:.1f}s')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Spectre
ax = axes[0]
x_eig = np.arange(1, N_SAMPLES + 1)
ax.semilogy(x_eig, spec_g, color='#e74c3c', lw=2, label=f'Global (eff. rank={er_g:.1f})')
ax.semilogy(x_eig, spec_l, color='#2980b9', lw=2, ls='--', label=f'Local p=2 (eff. rank={er_l:.1f})')
ax.semilogy(x_eig, spec_m, color='#27ae60', lw=2, ls='-.', label=f'Multi-echelles (eff. rank={er_m:.1f})')
ax.set_xlabel('Index valeur propre (ordre decroissant)')
ax.set_ylabel('Valeur propre normalisee (log)')
ax.set_title(f'Spectre de la matrice de Gram (Q={Q_SPEC}, Breast Cancer)\n'
              'Spectre plat = kernel expressif', fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim(0, N_SAMPLES)

# Spectre cumule
ax = axes[1]
ax.plot(x_eig, np.cumsum(spec_g), color='#e74c3c', lw=2, label='Global')
ax.plot(x_eig, np.cumsum(spec_l), color='#2980b9', lw=2, ls='--', label='Local p=2')
ax.plot(x_eig, np.cumsum(spec_m), color='#27ae60', lw=2, ls='-.', label='Multi-echelles')
for threshold in [0.9, 0.95, 0.99]:
    ax.axhline(threshold, color='grey', ls=':', lw=1, alpha=0.6)
    ax.text(N_SAMPLES * 0.98, threshold + 0.003, f'{int(threshold*100)}%',
            ha='right', fontsize=8, color='grey')
ax.set_xlabel('Nombre de composantes principales')
ax.set_ylabel('Variance cumulee')
ax.set_title('Spectre cumule\n(90% variance expliquee en combien de dims ?)', fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim(0, N_SAMPLES)

# Calcul du nb de dims pour 90%
for name, spec in [('Global', spec_g), ('Local p=2', spec_l), ('Multi-echelles', spec_m)]:
    n90 = int(np.searchsorted(np.cumsum(spec), 0.90)) + 1
    print(f'  {name:20s}: {n90} dims pour 90% variance')

plt.suptitle('A5 — Spectres des valeurs propres de Gram (Q=8, Breast Cancer, N=120)\n'
             'Rang effectif plus eleve = kernel plus expressif',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(OUT / '17_A5_spectra.png', dpi=150)
plt.show()
print('A5 sauvegarde')

---
## Synthese

Reproduction et extension des resultats de arXiv 2602.16097 sur donnees financieres.

In [ ]:
print('=' * 68)
print(' SYNTHESE — arXiv 2602.16097 sur donnees financieres')
print('=' * 68)

print('\n[A1] Concentration exponentielle du kernel global')
for ds_name, ds_label in DATASETS:
    p50_q4  = results_global[ds_name]['p50'][0]
    p50_q12 = results_global[ds_name]['p50'][-1]
    chute   = 100 * (p50_q12 - p50_q4) / (p50_q4 + 1e-24)
    print(f'  {ds_label:20s}: p50 Q=4={p50_q4:.4f} -> Q=12={p50_q12:.4f}  '
          f'({chute:+.0f}%)  [confirme concentration]')

print('\n[A2+A3] Mitigation par kernels locaux / multi-echelles (Q=12, Breast Cancer)')
qi_12 = D_VALUES.index(12)
for meth in methods_ms:
    p50_v = res_ms['breast_cancer'][meth]['p50'][qi_12]
    H_v   = res_ms['breast_cancer'][meth]['H'][qi_12]
    auc_v = res_ms['breast_cancer'][meth]['auc_m'][qi_12]
    print(f'  {labels_m[meth]:35s}: p50={p50_v:.4f}  H/H_max={H_v:.3f}  AUC={auc_v:.4f}')

print('\n[A4] Correlation mitigation geometrique -> AUC')
r_p50_auc = np.corrcoef(d_p50_all, d_auc_all)[0, 1]
r_cka_auc = np.corrcoef(cka_all, auc_all)[0, 1]
print(f'  corr(delta_p50, delta_AUC) = {r_p50_auc:+.4f}')
print(f'  corr(CKA,       AUC)       = {r_cka_auc:+.4f}')
if abs(r_p50_auc) < abs(r_cka_auc):
    print('  => CKA est un meilleur predicteur de l\'AUC que la mitigation geometrique')
    print('     Confirme la these principale du papier.')
else:
    print('  => Sur ces donnees, la mitigation geometrique correllee avec AUC.')
    print('     Resultat partiel — contexte dependant du dataset.')

print('\n[A5] Rang effectif (Q=8, Breast Cancer)')
print(f'  Global           : {er_g:.1f}')
print(f'  Local p=2        : {er_l:.1f}')
print(f'  Multi-echelles   : {er_m:.1f}')
print(f'  Gain rang effectif (multi vs global) : {100*(er_m-er_g)/er_g:+.1f}%')

print('\n' + '=' * 68)
print(' FIGURES GENEREES')
print('=' * 68)
for f in sorted(OUT.glob('17_*.png')):
    print(f'  {f.name}')